In [1]:
%pip install crewai crewai_tools langchain langchain_community langchain_ollama streamlit duckduckgo-search

Note: you may need to restart the kernel to use updated packages.


In [2]:
from crewai import LLM
# from langchain_ollama import OllamaLLM
# llm=LLM(
#     model="ollama/llama3.2",
#     base_url="http://localhost:11434"
# )

In [ ]:
%pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env into environment

from crewai import LLM

llm = LLM(
    model="groq/llama-3.1-8b-instant"
)

In [4]:
from crewai.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults 
import json

@tool("search_web_tool")
def search_web_tool(query: str) -> str:
    """
    Searches the web and returns results.
    Args:
        query: A plain text search query string. Example: 'best restaurants in Rome Italy'
    """
    # guard against LLM passing a dict
    if isinstance(query, dict):
        query = query.get("query", str(query))
    
    search_tool = DuckDuckGoSearchResults(num_results=10, verbose=True)
    return search_tool.run(query)

In [5]:
from crewai import Agent

# Agent Resercher
guide_expert= Agent( 
    role="City Local Guide Expert",
    goal="Provides information on things to do in the city based on the user's interests.",
    backstory="""A local expert with a passion for sharing the best experiences and hidden gems of their city.""",
    tools=[search_web_tool],
    verbose=True,
    max_iter=5,
    llm=LLM(model="ollama/llama3.2",base_url="http://localhost:11434"),  #ChatOpenAI(temperature=0, model="gpt-4o-mini"),
    allow_delegation=False,
    )

In [6]:
#Agent2 city expert

location_expert = Agent(
    role="Travel Trip Expert",
    goal="Adapt to the user destination vity language (French if city in French Country. Gather helpful information about to the city and city during travel.",
    backstory="""A seasoned traveler who has explored various destinations and knows the ins and outs of travel logistics.""",
    tools=[search_web_tool],
    verbose=True,
    max_iter=5,
    llm=LLM(model="ollama/llama3.2",base_url="http://localhost:11434"),
    allow_delegation=False,
    )


In [7]:
#agent 3 planer is main agent 
planner_expert = Agent(
    role="Travel Planning Expert",
    goal="Compiles all gathered information to provide a comprehensive travel plan.",
    backstory="""
    You are a professional guide with a passion for travel.
    An organizational wizard who can turn a list of possibilities into a seamless itinerary.
    """,
    tools=[search_web_tool],
    verbose=True,
    max_iter=5,
    llm=LLM(model="ollama/llama3.2",base_url="http://localhost:11434"),
    allow_delegation=False,
    )


In [8]:
from datetime import datetime
from crewai import Task

from_city = "India"
destination_city = "Rome"
date_from = "1st March 2025"
date_to = "7th March 2025"
interests = "sight seeing and good food"

location_task = Task(
    description=f"""
    In French : This task involves a comprehensive data collection process to provide the traveler with essential information about their destination. It includes researching and compiling details on various accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of living in the area. The task also covers transportation options, visa requirements, and any travel advisories that may be relevant.
    consider also the weather conditions forcast on the travel dates. and all the events that may be relevant to the traveler during the trip period.
    
    Traveling from : {from_city}
    Destination city : {destination_city}
    Arrival Date : {date_from}
    Departure Date : {date_to}

    Follow this rules : 
    1. if the {destination_city} is in a French country : Respond in FRENCH.
    """,
    expected_output=f"""
    if the {destination_city} is in a French country : Respond in FRENCH.
    In markdown format : A detailed markdown report that includes a curated list of recommended places to stay, a breakdown of daily living expenses, and practical travel tips to ensure a smooth journey.
    """,
    agent=location_expert,
    output_file='city_report.md',
)





In [9]:
guide_task = Task(
    description=f"""
    if the {destination_city} is in a French country : Respond in FRENCH.
    Tailored to the traveler's personal {interests}, this task focuses on creating an engaging and informative guide to the city's attractions. It involves identifying cultural landmarks, historical spots, entertainment venues, dining experiences, and outdoor activities that align with the user's preferences such {interests}. The guide also highlights seasonal events and festivals that might be of interest during the traveler's visit.
    Destination city : {destination_city}
    interests : {interests}
    Arrival Date : {date_from}
    Departure Date : {date_to}

    Follow this rules : 
    1. if the {destination_city} is in a French country : Respond in FRENCH.
    """,
    expected_output=f"""
    An interactive markdown report that presents a personalized itinerary of activities and attractions, complete with descriptions, locations, and any necessary reservations or tickets.
    """,

    agent=guide_expert,
    output_file='guide_report.md',
)


In [10]:
planner_task = Task(
    description=f"""
    This task synthesizes all collected information into a detaileds introduction to the city (description of city and presentation, in 3 pragraphes) cohesive and practical travel plan. and takes into account the traveler's schedule, preferences, and budget to draft a day-by-day itinerary. The planner also provides insights into the city's layout and transportation system to facilitate easy navigation.
    Destination city : {destination_city}
    interests : {interests}
    Arrival Date : {date_from}
    Departure Date : {date_to}

    Follow this rules : 
    1. if the {destination_city} is in a French country : Respond in FRENCH.
    """,
    expected_output="""
    if the {destination_city} is in a French country : Respond in FRENCH.
    A rich markdown document with emojis on each title and subtitle, that :
    In markdown format : 
    # Welcome to {destination_city} :
    A 4 paragraphes markdown formated including :
    - a curated articles of presentation of the city, 
    - a breakdown of daily living expenses, and spots to visit.
    # Here's your Travel Plan to {destination_city} :
    Outlines a daily detailed travel plan list with time allocations and details for each activity, along with an overview of the city's highlights based on the guide's recommendations
    """,
    context=[location_task, guide_task],
    #context=context,
    agent=planner_expert,
        output_file='travel_plan.md',
        )


In [11]:
# Task: Location
def make_location_task(agent, from_city, destination_city, date_from, date_to):
    return Task(
        description=f"""
        In French : This task involves a comprehensive data collection process to provide the traveler with essential information about their destination. It includes researching and compiling details on various accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of living in the area. The task also covers transportation options, visa requirements, and any travel advisories that may be relevant.
        consider also the weather conditions forcast on the travel dates. and all the events that may be relevant to the traveler during the trip period.
        
        Traveling from : {from_city}
        Destination city : {destination_city}
        Arrival Date : {date_from}
        Departure Date : {date_to}

        Follow this rules : 
        1. if the {destination_city} is in a French country : Respond in FRENCH.
        """,
        expected_output=f"""
        if the {destination_city} is in a French country : Respond in FRENCH.
        In markdown format : A detailed markdown report that includes a curated list of recommended places to stay, a breakdown of daily living expenses, and practical travel tips to ensure a smooth journey.
        """,
        agent=agent,
        output_file='city_report.md',
    )

# Task: Location
def make_guide_task(agent, destination_city, interests, date_from, date_to):    
    return Task(
        description=f"""
        if the {destination_city} is in a French country : Respond in FRENCH.
        Tailored to the traveler's personal {interests}, this task focuses on creating an engaging and informative guide to the city's attractions. It involves identifying cultural landmarks, historical spots, entertainment venues, dining experiences, and outdoor activities that align with the user's preferences such {interests}. The guide also highlights seasonal events and festivals that might be of interest during the traveler's visit.
        Destination city : {destination_city}
        interests : {interests}
        Arrival Date : {date_from}
        Departure Date : {date_to}

        Follow this rules : 
        1. if the {destination_city} is in a French country : Respond in FRENCH.
        """,
        expected_output=f"""
        An interactive markdown report that presents a personalized itinerary of activities and attractions, complete with descriptions, locations, and any necessary reservations or tickets.
        """,

        agent=agent,
        output_file='guide_report.md',
    )


# Task: Planner
def make_planner_task(context, agent, destination_city, interests, date_from, date_to):
    return Task(
        description=f"""
        This task synthesizes all collected information into a detaileds introduction to the city (description of city and presentation, in 3 pragraphes) cohesive and practical travel plan. and takes into account the traveler's schedule, preferences, and budget to draft a day-by-day itinerary. The planner also provides insights into the city's layout and transportation system to facilitate easy navigation.
        Destination city : {destination_city}
        interests : {interests}
        Arrival Date : {date_from}
        Departure Date : {date_to}

        Follow this rules : 
        1. if the {destination_city} is in a French country : Respond in FRENCH.
        """,
        expected_output=f"""
        if the {destination_city} is in a French country : Respond in FRENCH.
        A rich markdown document with emojis on each title and subtitle, that :
        In markdown format : 
        # Welcome to {destination_city} :
        A 4 paragraphes markdown formated including :
        - a curated articles of presentation of the city, 
        - a breakdown of daily living expenses, and spots to visit.
        # Here's your Travel Plan to {destination_city} :
        Outlines a daily detailed travel plan list with time allocations and details for each activity, along with an overview of the city's highlights based on the guide's recommendations
        """,
        #context=[location_task, guide_task],
        context=context,
        agent=agent,
        output_file='travel_plan.md',
        )


In [12]:
location_task = make_location_task(
  location_expert,
  from_city,
  destination_city,
  date_from,
  date_to
)

guide_task = make_guide_task(
  guide_expert,
  destination_city,
  interests,
  date_from,
  date_to
)

planner_task = make_planner_task(
  [location_task, guide_task],
  planner_expert,
  destination_city,
  interests,
  date_from,
  date_to,
)



In [ ]:
from crewai import Crew, Process

crew = Crew(
    agents=[location_expert, guide_expert, planner_expert],
    tasks=[location_task, guide_task, planner_task],
    process=Process.sequential,
    share_crew=False,
    verbose=True
)

result = crew.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ac3d7cf1-c28d-479d-98ed-62092778bf8c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          In French : This task involves a comprehensive data collection process to provide the traveler with    │
│  essential information about their destination. It includes researching and compiling details on various        │
│  accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of       │
│  living in the area. The task also covers transportation options, visa requirements, and any travel advisories  │
│  that may be relevant.                                                                                          │
│          consider also the weather conditions forcast on the travel dates. and all the events that may be       │
│  relevant to the traveler during the trip period.                                                               │
│                                                                                                                 │
│          Traveling from : India                                                                                 │
│          Destination city : Rome                                                                                │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│  ID: 19855be2-562c-4509-b5a0-73e8a1bd447a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Trip Expert                                                                                      │
│                                                                                                                 │
│  Task:                                                                                                          │
│          In French : This task involves a comprehensive data collection process to provide the traveler with    │
│  essential information about their destination. It includes researching and compiling details on various        │
│  accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of       │
│  living in the area. The task also covers transportation options, visa requirements, and any travel advisories  │
│  that may be relevant.                                                                                          │
│          consider also the weather conditions forcast on the travel dates. and all the events that may be       │
│  relevant to the traveler during the trip period.                                                               │
│                                                                                                                 │
│          Traveling from : India                                                                                 │
│          Destination city : Rome                                                                                │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_web_tool                                                                                          │
│  Args: {'query': '[Rome] budget-friendly accommodations + Rome Italy accommodations + cost of living in Rome    │
│  Italy + transportation options in Rome Italy + visa requirements for Indian citizens + travel advi...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

snippet: Plus, you ’ ll get a taste of Rome ’ s incredible food scene and get my list of the best places to stay in Rome depending on your budget., title: Rome in May 2025: The Definitive Guide | + Tips | Visit Italy, link: https://www.voyagetips.com/en/rome-in-may/, snippet: ... of the article, you will also find itineraries to visit Rome in 1, 2, 3, 4 or 5 days (or more!) as well as my suggestions of the best accommodations ..., title: 27 Best Things to Do in Rome | TOP Places to Visit | 2026, link: https://www.voyagetips.com/en/things-to-do-in-rome/, snippet: ... information on where to stay in Rome, the best area to stay in Rome, and accommodation options for any budget – from the best hotels in Rome to ..., title: Where To Stay In Rome: 7 Best Area To Stay In Rome, link: https://myadventuresacrosstheworld.com/where-to-stay-in-rome/, snippet: Filled with low-cost hotels and B&B’s, this is where to stay in Rome for budget travelers who don’t mind taking a short metro ride into the .

...

snippet: 3 weeks ago - Le Campo de’ Fiori accueille chaque matin un marché animé : arrivez avant 10h. La statue de Giordano Bruno en son centre rappelle que le philosophe y fut brûlé pour hérésie en 1600. Pour explorer le quartier à travers sa gastronomie, une visite culinaire Campo de’ Fiori + Trastevere est une bonne option., title: Visiter Rome : 10 incontournables à faire et voir (Italie), link: https://generationvoyage.fr/visiter-rome-faire-voir/, snippet: October 28, 2025 - Dans cet article, je vous propose un guide complet pour visiter Rome, avec tous mes conseils pratiques, bonnes adresses, expériences incontournables et idées pour organiser un séjour inoubliable. Que vous partiez pour un week-end à Rome, un premier voyage en Italie ou une escapade romantique, vous trouverez ici toutes les infos pour vivre Rome comme un vrai local !, title: Que faire à Rome: Les lieux incontournables à voir, link: https://www.mademoiselle-voyage.fr/europe-italie-que-faire-a-rome/, snippet: Marc

╭─────────────────────────────────────── ✅ Tool Execution Completed (#7) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_web_tool                                                                                          │
│  Output: snippet: 3 weeks ago - Le Campo de’ Fiori accueille chaque matin un marché animé : arrivez avant 10h.  │
│  La statue de Giordano Bruno en son centre rappelle que le philosophe y fut brûlé pour hérésie en 1600. Pour    │
│  explorer le quartier à travers sa gastronomie, une visite culinaire Campo de’ Fiori + Trastevere est une       │
│  bonne option., title: Visiter Rome : 10 incontournables à faire et voir (Italie), link:                        │
│  https://generationvoyage.fr/visiter-rome-faire-voir/, snippet: October 28, 2025 - Dans cet article, je vous    │
│  propose un guide complet pour visiter Rome, avec tous mes conseils pratiques, bonnes adresses, expériences     │
│  incontournables et idées pour organiser un séjour inoubliable. Que vous partiez pour un week-end à Rome, un    │
│  premier voyage en Italie ou une escapade romantique, vous trouverez ici toutes les infos pour vivre Rome       │
│  comme un vrai local !, title: Que faire à Rome: Les lieux incontournables à voir, link:                        │
│  https://www.mademoiselle-voyage.fr/europe-italie-que-faire-a-rome/, snippet: March 12, 2026 - N’hésitez pas à  │
│  vous poser près de la fontaine de la Barcaccia pour profiter de l’ambiance animée de la place tout en          │
│  dégustant une bonne glace 😉 ... Il est interdit de s’asseoir sur les escaliers de la Trinité-des-Monts sous   │
│  peine d’une lourde amende. La Piazza di Spana est entourée de boutiques de grandes marques et d’hôtels de      │
│  luxe et il n’est donc pas rare de pouvoir apercevoir des célébrités dans ce coin de Rome 😉, title: Mes        │
│  conseils pour visiter Rome en 3 jours : itinéraire et visites | BLOG VOYAGE, link:                             │
│  https://www.elovoyage.com/visiter-rome-en-3-jours/, snippet: October 23, 2025 - Sinon, vous avez toujours      │
│  l’option simple mais efficace de la glace (gelato en italien), aux parfums aussi savoureux que variés.         │
│  Giolitti, situé sur la Via Uffici del Vicario vers le Panthéon, est une très bonne adresse. Prix d’activités   │
│  ..., title: Rome : TOP 10 activités et visites - Que faire, que découvrir ?, link:                             │
│  https://www.que-faire-en-voyage.com/que-faire-a-rome-top-10-activites/, snippet: December 22, 2025 -           │
│  N’oubliez pas de passer à la belle Piazza di Santa Maria avant de profiter de l’ambiance festive du soir dans  │
│  ce quartier. Le Trastevere est le meilleur quartier pour manger des spécialités italiennes à un prix           │
│  raisonnable : les bonnes ..., title: Visiter Rome en 4 jours : programme détaillé pour ne rien rater !, link:  │
│  https://www.partir-a-rome.com/planning/rome-4-jours, snippet: February 18, 2026 - Les couleurs usées des       │
│  façades, les ruelles sinueuses, le linge pendant aux balcons, les vespas garées négligemment, vous trouverez   │
│  tous les clichés qui participent au mythe italien. Errer dans ce quartier est certainement la meilleure chose  │
│  à faire. Des rues comme Via del Moro, Vicolo del cinque, Vicolo del Bologna, sont très représentatives et      │
│  immanquables. On est bien loin de l’agitation du Vatican ou du centre… Ici, artisans, restaurants et petits    │
│  commerces font vivre ce quartier pétillant et plein de vie. Le cœur de ce quartier si attachant, c’est         │
│  l’historique Piazza di Santa Maria., title: Nos bonnes adresses à Rome | Little Weekends, link:                │
│  https://littleweekends.fr/guides-voyage-europe/rome/, sn

Tool search_web_tool executed with result (from cache): snippet: 3 weeks ago - Le Campo de’ Fiori accueille chaque matin un marché animé : arrivez avant 10h. La statue de Giordano Bruno en son centre rappelle que le philosophe y fut brûlé pour hérésie en 1...


╭──────────────────────────────────────── 🔧 Tool Execution Started (#8) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_web_tool                                                                                          │
│  Args: {'query': 'meilleures activites de sight-seeing et bonnes addresses culinaires dans Rome Italie pour le  │
│  1er Mars - 7 mars 2025'}                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#8) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_web_tool                                                                                          │
│  Output: snippet: 3 weeks ago - Le Campo de’ Fiori accueille chaque matin un marché animé : arrivez avant 10h.  │
│  La statue de Giordano Bruno en son centre rappelle que le philosophe y fut brûlé pour hérésie en 1600. Pour    │
│  explorer le quartier à travers sa gastronomie, une visite culinaire Campo de’ Fiori + Trastevere est une       │
│  bonne option., title: Visiter Rome : 10 incontournables à faire et voir (Italie), link:                        │
│  https://generationvoyage.fr/visiter-rome-faire-voir/, snippet: October 28, 2025 - Dans cet article, je vous    │
│  propose un guide complet pour visiter Rome, avec tous mes conseils pratiques, bonnes adresses, expériences     │
│  incontournables et idées pour organiser un séjour inoubliable. Que vous partiez pour un week-end à Rome, un    │
│  premier voyage en Italie ou une escapade romantique, vous trouverez ici toutes les infos pour vivre Rome       │
│  comme un vrai local !, title: Que faire à Rome: Les lieux incontournables à voir, link:                        │
│  https://www.mademoiselle-voyage.fr/europe-italie-que-faire-a-rome/, snippet: March 12, 2026 - N’hésitez pas à  │
│  vous poser près de la fontaine de la Barcaccia pour profiter de l’ambiance animée de la place tout en          │
│  dégustant une bonne glace 😉 ... Il est interdit de s’asseoir sur les escaliers de la Trinité-des-Monts sous   │
│  peine d’une lourde amende. La Piazza di Spana est entourée de boutiques de grandes marques et d’hôtels de      │
│  luxe et il n’est donc pas rare de pouvoir apercevoir des célébrités dans ce coin de Rome 😉, title: Mes        │
│  conseils pour visiter Rome en 3 jours : itinéraire et visites | BLOG VOYAGE, link:                             │
│  https://www.elovoyage.com/visiter-rome-en-3-jours/, snippet: October 23, 2025 - Sinon, vous avez toujours      │
│  l’option simple mais efficace de la glace (gelato en italien), aux parfums aussi savoureux que variés.         │
│  Giolitti, situé sur la Via Uffici del Vicario vers le Panthéon, est une très bonne adresse. Prix d’activités   │
│  ..., title: Rome : TOP 10 activités et visites - Que faire, que découvrir ?, link:                             │
│  https://www.que-faire-en-voyage.com/que-faire-a-rome-top-10-activites/, snippet: December 22, 2025 -           │
│  N’oubliez pas de passer à la belle Piazza di Santa Maria avant de profiter de l’ambiance festive du soir dans  │
│  ce quartier. Le Trastevere est le meilleur quartier pour manger des spécialités italiennes à un prix           │
│  raisonnable : les bonnes ..., title: Visiter Rome en 4 jours : programme détaillé pour ne rien rater !, link:  │
│  https://www.partir-a-rome.com/planning/rome-4-jours, snippet: February 18, 2026 - Les couleurs usées des       │
│  façades, les ruelles sinueuses, le linge pendant aux balcons, les vespas garées négligemment, vous trouverez   │
│  tous les clichés qui participent au mythe italien. Errer dans ce quartier est certainement la meilleure chose  │
│  à faire. Des rues comme Via del Moro, Vicolo del cinque, Vicolo del Bologna, sont très représentatives et      │
│  immanquables. On est bien loin de l’agitation du Vatican ou du centre… Ici, artisans, restaurants et petits    │
│  commerces font vivre ce quartier pétillant et plein de vie. Le cœur de ce quartier si attachant, c’est         │
│  l’historique Piazza di Santa Maria., title: Nos bonnes adresses à Rome | Little Weekends, link:                │
│  https://littleweekends.fr/guides-voyage-europe/rome/, sn